# 5 - Hiérarchies et libellés de colonnes

Deux mécanismes de `metadata` reposent sur la même logique : un pointeur nullable
(`parent_name` pour une hiérarchie, `label_for` pour un libellé), validé à
l'écriture, sans aucune table auxiliaire, lu directement dans la fact table par
`SELECT DISTINCT`. Ce notebook les illustre ensemble : les hiérarchies de colonnes
d'abord (§2.5 de `specification-bdd.md`), les colonnes de libellés ensuite (§2.6).

## Hiérarchies de colonnes

Une hiérarchie de menu (group-options, arbre de sélection de profondeur arbitraire)
est **une chaîne de colonnes** de la `fact_table`, déclarée dans `metadata.parent_name`
(cf. `specification-bdd.md`, §2.5). Il n'existe **aucune table auxiliaire** : la
parente de `commune` est `departement`, la parente de `departement` est `region`, et
la hiérarchie des *valeurs* est déjà dans la fact table
(`SELECT DISTINCT region, departement, commune`).

Invariants validés à l'écriture :

- la colonne parente existe (dans le DataFrame à la construction, dans `metadata` lors
  d'une correction) ;
- le graphe des `parent_name` est une **forêt** (pas de cycle, une seule parente par
  colonne), détecté par un parcours de proche en proche ;
- une colonne d'une hiérarchie est **catégorielle** ; si elle ne l'est pas par le
  seuil, elle est **forcée** à `True` avec un avertissement, comme le ferait
  `categorical_overrides`.

**Convention pour un arbre irrégulier** (feuilles à des profondeurs différentes) : les
niveaux absents sont `NULL`. Le constructeur d'arbre s'arrête au premier niveau `NULL`
et ne répète jamais la valeur du niveau supérieur (cela ferait apparaître un faux
nœud).

### Table des matières

**Hiérarchies de colonnes**

0. [Importation des modules](#s0)
1. [Données synthétiques : une hiérarchie géographique irrégulière](#s1)
2. [Déclarer la hiérarchie via `hierarchies` à la construction](#s2)
   - [Forçage catégoriel d'une colonne de hiérarchie](#s2_1)
3. [Alternative : `parent_name` via `column_metadata`](#s3)
4. [Reconstituer les chaînes avec `get_column_hierarchies`](#s4)
5. [Construire un arbre de menu par `SELECT DISTINCT`](#s5)
6. [Corriger la hiérarchie sur une base existante](#s6)
7. [Supprimer une colonne parente : refus, puis `cascade=True`](#s7)
8. [Cas d'erreur validés à l'écriture](#s8)

**Codes et libellés**

9. [Données synthétiques : un extrait de nomenclature douanière (nc6 → nc8)](#s9)
10. [Déclarer la hiérarchie et les libellés à la construction](#s10)
11. [Lire la correspondance code → libellé par `SELECT DISTINCT`](#s11)
12. [Arbre de menu (code, libellé) par niveau](#s12)
13. [Agrégat par code avec `ANY_VALUE(libellé)`](#s13)
14. [Révision de nomenclature : l'upsert est refusé](#s14)
15. [Correction du libellé par `update_value_labels`](#s15)
16. [Time travel : l'ancien libellé reste lisible](#s16)

## 0. Importation des modules <a id="s0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
import warnings

import narwhals as nw
import polars as pl

# Ajout du chemin vers le package
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.operations import DatabaseDeleter, DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder, SchemaBuilder
from dt_ducklake_manager.utils import get_column_hierarchies, get_value_label_columns

## 1. Données synthétiques : une hiérarchie géographique irrégulière <a id="s1"></a>

Trois niveaux de colonnes : `region` -> `departement` -> `commune`. La dernière ligne
illustre un arbre **irrégulier** : ce département n'a pas de commune renseignée
(`commune = None`), comme le prévoit la convention `NULL`.

In [ ]:
df = pl.DataFrame(
    {
        "id": [1, 2, 3, 4, 5],
        "date": ["2026-01-01"] * 5,
        "region": [
            "Bretagne",
            "Bretagne",
            "Bretagne",
            "Ile-de-France",
            "Ile-de-France",
        ],
        "departement": [
            "Finistere",
            "Finistere",
            "Morbihan",
            "Paris",
            "Essonne",
        ],
        "commune": ["Brest", "Quimper", "Vannes", "Paris", None],
        "value": [12.5, 8.1, 6.3, 40.2, 5.0],
    }
)
df

## 2. Déclarer la hiérarchie via `hierarchies` à la construction <a id="s2"></a>

`SchemaBuilder` / `DuckLakeTablesBuilder` acceptent un paramètre dédié
`hierarchies: dict[str, str]` (colonne enfant -> colonne parente). Il est validé et
écrit dans `metadata.parent_name` en une seule fois, sans table auxiliaire.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    builder = DuckLakeTablesBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "departement", "departement": "region"},
        dataset_label="Consommation régionale",
    )

builder.build_schema()

builder.conn.execute(
    "SELECT name, parent_name, is_categorical FROM metadata ORDER BY name"
).pl()

`region`, `departement` et `commune` portent chacune le nom de leur colonne parente
(`NULL` pour `region`, la racine). Les trois colonnes sont déjà catégorielles sous ce
seuil.

### Forçage catégoriel d'une colonne de hiérarchie <a id="s2_1"></a>

Avec un seuil plus bas, `commune` (3 modalités) resterait catégorielle mais
`departement` franchirait le seuil s'il avait plus de deux modalités. Voici le cas où
une colonne de la hiérarchie n'est **pas** catégorielle par le seuil : elle est forcée
à `True`, exactement comme `categorical_overrides` le ferait.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    forced_builder = SchemaBuilder(
        df,
        categorical_threshold=1,  # aucune colonne texte n'est catégorielle par seuil
        primary_keys=["id"],
        hierarchies={"commune": "departement", "departement": "region"},
    )
    forced_metadata = forced_builder.create_metadata_table()

print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
forced_metadata.filter(nw.col("name").is_in(["region", "departement", "commune"]))[
    ["name", "is_categorical"]
]

## 3. Alternative : `parent_name` via `column_metadata` <a id="s3"></a>

`parent_name` peut aussi être renseigné colonne par colonne dans `column_metadata`
(prompt 3), au même titre que `unit` ou `default_aggregation`. Les deux sources
doivent être **cohérentes** : une contradiction lève une `ValueError` explicite.

In [ ]:
# Équivalent au paramètre hierarchies, exprimé via column_metadata
alt_builder = SchemaBuilder(
    df,
    categorical_threshold=10,
    primary_keys=["id"],
)
alt_metadata = alt_builder.create_metadata_table(
    column_metadata={"commune": {"parent_name": "departement"}}
)
alt_metadata.filter(nw.col("name") == "commune")[["name", "parent_name"]]

In [ ]:
# hierarchies et column_metadata en désaccord sur la parente de 'commune'
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "region"},
    ).create_metadata_table(column_metadata={"commune": {"parent_name": "departement"}})
except ValueError as exc:
    print("hiérarchies contradictoires ->", exc)

## 4. Reconstituer les chaînes avec `get_column_hierarchies` <a id="s4"></a>

Fonction de référence pour les couches API : elle relit `metadata.parent_name` et
retourne une liste de chaînes racine -> feuille, une par colonne feuille.

In [ ]:
get_column_hierarchies(builder.conn, schema=builder.schema)

## 5. Construire un arbre de menu par `SELECT DISTINCT` <a id="s5"></a>

La hiérarchie des *valeurs* est déjà dans la fact table : un `SELECT DISTINCT` sur la
chaîne de colonnes suffit à obtenir l'arbre complet, sans aucune jointure. Convention
pour l'arbre irrégulier : on s'arrête au premier niveau `NULL` et on ne répète jamais
la valeur du niveau supérieur.

In [ ]:
chain = get_column_hierarchies(builder.conn, schema=builder.schema)[0]
rows = builder.conn.execute(
    f"SELECT DISTINCT {', '.join(chain)} FROM {builder._qualified('fact_table')}"
    f" ORDER BY {', '.join(chain)}"
).fetchall()


def build_menu_tree(chain: list[str], rows: list[tuple]) -> dict:
    """Build a nested {label: subtree} menu from root-to-leaf DISTINCT rows.

    Stops at the first NULL level in a row and never repeats the parent's own
    label as a fake child node.
    """
    tree: dict = {}
    for row in rows:
        node = tree
        for value in row:
            if value is None:
                break
            node = node.setdefault(value, {})
    return tree


build_menu_tree(chain, rows)

La branche `Ile-de-France -> Essonne` s'arrête bien à `departement` : aucun faux nœud
`None` ni répétition de `"Essonne"` au niveau `commune`.

## 6. Corriger la hiérarchie sur une base existante <a id="s6"></a>

`update_column_metadata(column, parent_name=...)` déclare ou corrige un lien de
hiérarchie sans reconstruire la base. Il valide l'existence de la colonne parente dans
`metadata` et l'absence de cycle sur le graphe courant, et force catégorielles les deux
extrémités du lien si besoin.

In [ ]:
updater = DatabaseUpdater(connection=builder.conn, categorical_threshold=10)

# 'value' n'est pas catégorielle : la déclarer enfant de 'commune' force son statut
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    updater.update_column_metadata("value", parent_name="commune")

print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
builder.conn.execute(
    "SELECT name, parent_name, is_categorical FROM metadata"
    " WHERE name IN ('value', 'commune')"
).pl()

In [ ]:
# Une modification créant un cycle est refusée (commune -> value -> commune)
try:
    updater.update_column_metadata("commune", parent_name="value")
except ValueError as exc:
    print("cycle refusé ->", exc)

## 7. Supprimer une colonne parente : refus, puis `cascade=True` <a id="s7"></a>

`DatabaseDeleter.delete_columns` refuse par défaut de supprimer une colonne qui est la
parente d'une autre colonne : cela orphelinerait la hiérarchie. `cascade=True` autorise
la suppression et détache les enfants (`parent_name` mis à `NULL`, avec un
avertissement).

In [ ]:
deleter = DatabaseDeleter(connection=builder.conn)

# Refus par défaut : 'departement' est la parente de 'commune'
result = deleter.delete_columns(["departement"], use_transaction=False)
print("sans cascade ->", result)

In [ ]:
# cascade=True : suppression autorisée, 'commune' est détachée
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = deleter.delete_columns(
        ["departement"], use_transaction=False, cascade=True
    )

print("avec cascade ->", result)
print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
builder.conn.execute(
    "SELECT name, parent_name FROM metadata WHERE name = 'commune'"
).pl()

## 8. Cas d'erreur validés à l'écriture <a id="s8"></a>

In [ ]:
# 8.1 Auto-référence (une colonne ne peut pas être sa propre parente)
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"region": "region"},
    ).create_metadata_table()
except ValueError as exc:
    print("auto-référence         ->", exc)

# 8.2 Cycle à deux colonnes (A -> B -> A)
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"region": "departement", "departement": "region"},
    ).create_metadata_table()
except ValueError as exc:
    print("cycle à deux colonnes   ->", exc)

# 8.3 Colonne parente inexistante dans le DataFrame
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "not_a_column"},
    )
except ValueError as exc:
    print("parente inexistante     ->", exc)

# 8.4 update_column_metadata : parente absente de la table metadata
try:
    updater.update_column_metadata("value", parent_name="not_a_column")
except ValueError as exc:
    print("parente absente (update) ->", exc)

## Codes et libellés

Certaines colonnes portent un **code métier** (nomenclature tarifaire, code pays, code
INSEE) qui doit être restitué tel quel, associé à un **libellé** lisible porté par une
**autre colonne** de la fact table. Le lien est déclaré dans `metadata.label_for`, **sur
la colonne de libellés**, qui pointe vers la colonne de code (cf.
`specification-bdd.md`, §2.6) — même forme que `parent_name` : l'enfant (ici le
libellé) pointe vers sa cible, ce qui permet à **plusieurs** colonnes de libellés de
pointer vers le même code (langues, libellé court/long).

## 9. Données synthétiques : un extrait de nomenclature douanière (nc6 → nc8) <a id="s9"></a>

Deux niveaux de codes (`nc6` -> `nc8`, une hiérarchie de colonnes comme au §1-8), et des
libellés : `nc8` en porte deux (`nc8_libelle_fr`, `nc8_libelle_en`), `nc6` un seul
(`nc6_libelle`). `pays_origine` est un code **sans** colonne de libellés — tous les
codes n'en ont pas besoin. La dernière ligne illustre un `nc8` **sans libellé**
(`NULL`), cohérent avec la dépendance fonctionnelle (un code non libellé reste valide,
tant que *toutes* ses lignes sont `NULL`).

In [ ]:
labels_df = pl.DataFrame(
    {
        "id": [1, 2, 3, 4, 5, 6],
        "date": [
            "2026-01-01",
            "2026-01-01",
            "2026-01-02",
            "2026-01-02",
            "2026-01-03",
            "2026-01-03",
        ],
        "nc6": ["010121", "010129", "010129", "010130", "010130", "010121"],
        "nc8": [
            "01012100",
            "01012910",
            "01012990",
            "01013000",
            "01019000",
            "01012100",
        ],
        "nc8_libelle_fr": [
            "Chevaux reproducteurs de race pure",
            "Chevaux destinés à la boucherie",
            "Chevaux, autres",
            "Ânes",
            None,
            "Chevaux reproducteurs de race pure",
        ],
        "nc8_libelle_en": [
            "Pure-bred breeding horses",
            "Horses for slaughter",
            "Horses, other",
            "Asses",
            None,
            "Pure-bred breeding horses",
        ],
        "nc6_libelle": [
            "Chevaux reproducteurs",
            "Chevaux, autres",
            "Chevaux, autres",
            "Ânes, mulets et bardots",
            "Ânes, mulets et bardots",
            "Chevaux reproducteurs",
        ],
        "pays_origine": ["FR", "DE", "DE", "ES", "IT", "FR"],
        "valeur": [15000.0, 8200.0, 3100.0, 500.0, 200.0, 9800.0],
    }
)
labels_df

## 10. Déclarer la hiérarchie et les libellés à la construction <a id="s10"></a>

`hierarchies` déclare `nc8 -> nc6` comme au §2 ; `value_labels` (mapping colonne de
libellés -> colonne de code) déclare les trois liens en une fois. Un vrai catalogue
DuckLake est nécessaire ici (et non plus une connexion `:memory:`) : les sections
suivantes utilisent `update_value_labels` et le time travel, qui exigent un catalogue
réel.

In [ ]:
# Suppression du catalogue et des données existants pour garantir un état initial propre
CATALOG_PATH = os.path.join("../outputs", "value_labels_demo.ducklake")
DATA_PATH = os.path.join("../outputs", "value_labels_demo_data/")
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)

labels_conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()

labels_builder = DuckLakeTablesBuilder(
    labels_df,
    categorical_threshold=20,
    primary_keys=["id"],
    hierarchies={"nc8": "nc6"},
    value_labels={
        "nc8_libelle_fr": "nc8",
        "nc8_libelle_en": "nc8",
        "nc6_libelle": "nc6",
    },
    connection=labels_conn,
    dataset_label="Extrait de nomenclature douanière",
)
build_report = labels_builder.build_schema(run_id="build-nc8")
print(build_report.summary())

# Snapshot de référence : capturé juste après la construction, avant relabellisation
snapshot_after_build = build_report.snapshot_after

labels_builder.conn.execute(
    "SELECT name, parent_name, label_for, is_categorical FROM metadata ORDER BY name"
).pl()

`nc8_libelle_fr`, `nc8_libelle_en` et `nc6_libelle` portent chacune un `label_for` non
nul ; `pays_origine` n'apparaît dans aucune des deux colonnes (`parent_name` et
`label_for` sont `NULL` pour elle) — un code n'a besoin de ni l'un ni l'autre. Le
build a accepté `01019000` avec un libellé `NULL` sur toutes ses lignes : la dépendance
fonctionnelle est respectée (aucune valeur concurrente).

## 11. Lire la correspondance code → libellé par `SELECT DISTINCT` <a id="s11"></a>

`get_value_label_columns` relit `metadata.label_for` et retourne, pour chaque colonne de
code, la liste triée de ses colonnes de libellés — implémentation de référence pour
l'API, symétrique de `get_column_hierarchies`. La correspondance elle-même n'a besoin
d'aucune jointure : un `SELECT DISTINCT` sur la fact table suffit.

In [ ]:
label_columns = get_value_label_columns(
    labels_builder.conn, schema=labels_builder.schema
)
print(label_columns)

# Correspondance nc8 -> libellé (FR et EN), NULL compris pour le code non libellé
fact_table = labels_builder._qualified("fact_table")
labels_builder.conn.execute(
    f"SELECT DISTINCT nc8, nc8_libelle_fr, nc8_libelle_en FROM {fact_table}"
    " ORDER BY nc8"
).pl()

## 12. Arbre de menu (code, libellé) par niveau <a id="s12"></a>

Hiérarchie de colonnes (`get_column_hierarchies`) et libellés (`get_value_label_columns`)
se combinent directement : pour chaque niveau de la chaîne, on lit le code **et** son
libellé (français, ici) dans le même `SELECT DISTINCT`, sans jointure.

In [ ]:
chain = get_column_hierarchies(labels_builder.conn, schema=labels_builder.schema)[0]
# Un seul libellé (français) retenu par niveau, pour la lisibilité de l'arbre
level_label = {"nc6": "nc6_libelle", "nc8": "nc8_libelle_fr"}
select_cols = [c for level in chain for c in (level, level_label[level])]

rows = labels_builder.conn.execute(
    f"SELECT DISTINCT {', '.join(select_cols)} FROM {fact_table}"
    f" ORDER BY {', '.join(chain)}"
).fetchall()


def build_labelled_menu_tree(chain: list[str], rows: list[tuple]) -> dict:
    """Build a nested {(code, label): subtree} menu from root-to-leaf DISTINCT rows.

    Each row carries (code, label) pairs, two columns per level. Stops at the
    first NULL code, exactly like the plain-hierarchy tree of section 5.
    """
    tree: dict = {}
    for row in rows:
        node = tree
        for i in range(0, len(row), 2):
            code, label = row[i], row[i + 1]
            if code is None:
                break
            node = node.setdefault((code, label), {})
    return tree


build_labelled_menu_tree(chain, rows)

## 13. Agrégat par code avec `ANY_VALUE(libellé)` <a id="s13"></a>

`ANY_VALUE` est licite ici précisément grâce à la dépendance fonctionnelle code ->
libellé : toutes les lignes d'un même `nc8` portent le même `nc8_libelle_fr` (ou
toutes `NULL`), donc n'importe laquelle convient pour l'agrégat.

In [ ]:
labels_builder.conn.execute(f"""
    SELECT nc8, ANY_VALUE(nc8_libelle_fr) AS libelle, SUM(valeur) AS total
    FROM {fact_table}
    GROUP BY nc8
    ORDER BY nc8
""").pl()

## 14. Révision de nomenclature : l'upsert est refusé <a id="s14"></a>

Une révision qui renomme un code viole la dépendance fonctionnelle si elle arrive par
upsert : les lignes déjà écrites gardent l'ancien libellé, la nouvelle ligne porte le
nouveau — deux libellés concurrents pour le même `nc8`. `update_database` le refuse,
**dans la transaction** : rien n'est écrit, ni le libellé ni la nouvelle ligne.

In [ ]:
labels_updater = DatabaseUpdater(
    connection=labels_builder.conn,
    schema=labels_builder.schema,
    categorical_threshold=20,
)

# Compte de lignes et libellé courant avant la tentative, pour vérifier ensuite que
# rien n'a bougé
rows_before = labels_builder.conn.execute(
    f"SELECT COUNT(*) FROM {fact_table}"
).fetchone()[0]
label_before = labels_builder.conn.execute(
    f"SELECT DISTINCT nc8_libelle_fr FROM {fact_table} WHERE nc8 = '01012100'"
).fetchall()

revision_df = pl.DataFrame(
    {
        "id": [7],
        "date": ["2026-01-04"],
        "nc6": ["010121"],
        "nc8": ["01012100"],
        "nc8_libelle_fr": ["Chevaux de race pure (nouvelle appellation)"],
        "nc8_libelle_en": ["Pure-bred horses (revised)"],
        "nc6_libelle": ["Chevaux reproducteurs"],
        "pays_origine": ["FR"],
        "valeur": [500.0],
    }
)

try:
    labels_updater.update_database(update_df=revision_df)
except ValueError as exc:
    print("upsert refusé ->", exc)

rows_after = labels_builder.conn.execute(
    f"SELECT COUNT(*) FROM {fact_table}"
).fetchone()[0]
label_after = labels_builder.conn.execute(
    f"SELECT DISTINCT nc8_libelle_fr FROM {fact_table} WHERE nc8 = '01012100'"
).fetchall()

print(f"Lignes avant / après : {rows_before} / {rows_after}")
print(f"Libellé(s) avant / après : {label_before} / {label_after}")
assert rows_before == rows_after
assert label_before == label_after

## 15. Correction du libellé par `update_value_labels` <a id="s15"></a>

La seule façon légitime de relabelliser un code : un unique `UPDATE … FROM` réécrit
**toutes** les lignes des codes donnés, dans une transaction. Un code absent de la base
est signalé (`report.warnings`), jamais inséré — variante d'argument montrée ici avec
un code fictif `99999999`.

In [ ]:
new_labels_fr = pl.DataFrame(
    {
        "nc8": ["01012100", "99999999"],
        "nc8_libelle_fr": [
            "Chevaux de race pure (nouvelle appellation)",
            "Code inconnu",
        ],
    }
)

relabel_report = labels_updater.update_value_labels(
    "nc8_libelle_fr", new_labels_fr, run_id="relabel-nc8-01012100"
)
print(relabel_report.summary())
print("warnings ->", relabel_report.warnings)

labels_builder.conn.execute(
    f"SELECT DISTINCT nc8, nc8_libelle_fr FROM {fact_table} WHERE nc8 = '01012100'"
).pl()

## 16. Time travel : l'ancien libellé reste lisible <a id="s16"></a>

Le libellé affiché est toujours le **courant**, mais l'ancien reste lisible par time
travel DuckLake — `snapshot_after_build`, capturé juste après la construction (§10),
précède la relabellisation du §15. DuckLake interdit d'attacher deux fois le même
fichier catalogue dans le même processus : `at_clause()` génère la clause SQL `AT
(VERSION => n)` à utiliser directement sur la connexion courante, sans rouvrir de
connexion.

In [ ]:
connector_past = DuckLakeConnector(
    CATALOG_PATH, DATA_PATH, snapshot_version=snapshot_after_build
)
at = connector_past.at_clause()

past = labels_builder.conn.execute(
    f"SELECT DISTINCT nc8, nc8_libelle_fr FROM fact_table {at} WHERE nc8 = '01012100'"
).pl()
current = labels_builder.conn.execute(
    "SELECT DISTINCT nc8, nc8_libelle_fr FROM fact_table WHERE nc8 = '01012100'"
).pl()

print(f"Libellé au snapshot {snapshot_after_build} (avant relabellisation) :")
print(past)
print("Libellé courant (après update_value_labels) :")
print(current)